<a href="https://colab.research.google.com/github/samilnamli/transformer_experiments/blob/colab/voxpopuli-main-results/notebooks/colab/main_results_voxpopuli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoxPopuli — Whisper correction (Colab)

Runs the **Whisper re-decode** that fixes VoxPopuli's broken Whisper WER
(raw ~121% → ~18% median), and — optionally — the full main-results
comparison table.

Uses a Blackwell-compatible torch (cu128) so it runs on the RTX PRO 6000.
Run the cells top to bottom.

## 1. Setup — clone & install

In [ ]:
%cd /content
!pip -q install uv
!git clone -b colab/voxpopuli-main-results https://github.com/samilnamli/transformer_experiments.git
%cd /content/transformer_experiments
!uv sync
!uv run python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0))"

## 2. HuggingFace token

Add `HF_TOKEN` under Colab **🔑 Secrets** (left sidebar).

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## 3. Download data (~24 GB)

In [ ]:
!uv run python -c "from src.data.downloads import download_drive_file as d; "\
  "d('1yf-G-DWhhZLlqeGXZ77GbuhTBmhqyyXA', 'configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet')"

## 4. Whisper correction (re-decode)

Writes the corrected overrides `.npz`. Takes ~1 h on the GPU.

In [ ]:
!uv run python scripts/redecode_voxpopuli_whisper_overrides_from_features.py \
  --input-parquet configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet \
  --output-npz data/processed/facebook_voxpopuli/whisper_decode_overrides_from_features.npz

## 5. Cache data to HuggingFace Hub (run once)

Uploads the parquet **and** the corrected Whisper `.npz` to
`huseyin-karaca/hit-asr` (a dataset repo). After this, future runs can
skip the Google Drive download *and* the 1 h re-decode entirely — just
pull both files from the Hub (snippet in the next cell's comment).

> Needs an `HF_TOKEN` with **write** access.

In [ ]:
import os
from huggingface_hub import HfApi

REPO = 'huseyin-karaca/hit-asr'
FILES = {
    'configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet':
        'combined_features_with_transcripts.parquet',
    'data/processed/facebook_voxpopuli/whisper_decode_overrides_from_features.npz':
        'whisper_decode_overrides_from_features.npz',
}
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(REPO, repo_type='dataset', exist_ok=True)
for local, name in FILES.items():
    print('uploading', name)
    api.upload_file(path_or_fileobj=local, path_in_repo=name,
                    repo_id=REPO, repo_type='dataset')
print('done ->', 'https://huggingface.co/datasets/' + REPO)

# Future runs — replace sections 3+4 with this to skip Drive + re-decode:
#   from huggingface_hub import hf_hub_download
#   hf_hub_download(REPO, 'combined_features_with_transcripts.parquet', repo_type='dataset',
#                   local_dir='configs/data/processed/facebook_voxpopuli')
#   hf_hub_download(REPO, 'whisper_decode_overrides_from_features.npz', repo_type='dataset',
#                   local_dir='data/processed/facebook_voxpopuli')

## 6. (Optional) Full comparison table

20 fun seeds by default. Trim with `experiment.seeds` for a quick pass.
Aggregation + significance tests run automatically at the end.

In [ ]:
!uv run python run.py experiment=main_results_voxpopuli_colab \
  experiment.data.batch_size=256 \
  experiment.seeds=[42,1337,73]